In [2]:
"""
v4 Multi-Video Autoencoder Pipeline — Option A training on multiple videos
================================================
Trains one dual model (raw + diff) per video independently.
Evaluates each video separately then prints a comparison table.
GPU-ready: uses CUDA if available, falls back to MPS/CPU.

Usage:
  python v4_multi_video.py

Outputs per video:
  outputs/v4_multi/{VIDEO_NAME}/
    model_A_raw_final.pth
    model_B_diff_final.pth
    loss_curve.png
    results.png
    weight_sweep.png
    suspicious_frames/
    config.txt

Final output:
  outputs/v4_multi/comparison_table.csv
  outputs/v4_multi/comparison_plot.png
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
import shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (roc_auc_score, precision_recall_fscore_support,
                             confusion_matrix, average_precision_score)
from tqdm import tqdm
from datetime import datetime
import cv2

In [3]:


# ══════════════════════════════════════════════════════════════
# CONFIG — update paths for RenkuLab
# ══════════════════════════════════════════════════════════════
# Base folder containing one subfolder per video
# Each subfolder should be named exactly as the video appears in the CSV
# e.g. data/frames/FH102_02/frame_00000.jpg

FRAMES_BASE    = "/home/renku/work/frames"
CSV_PATH       = "/home/renku/work/videos/Weinstein2018MEE_ground_truth.csv"
OUTPUT_BASE    = "/home/renku/work/camera-trap-gan/outputs/v5_multi"
TIMESTAMP      = datetime.now().strftime("%Y%m%d_%H%M%S")

# Model hyperparameters
IMG_SIZE       = 256
BATCH_SIZE     = 64      # increased for GPU
EPOCHS         = 30      # full training with GPU
LEARNING_RATE  = 0.001
LATENT_DIM     = 256
THRESHOLD_STD  = 2
SMOOTH_WINDOW  = 2
WEIGHT_RAW     = 0.6     # from weight sweep results — raw model better
WEIGHT_DIFF    = 0.4

# Videos to process — must match subfolder names AND CSV Video column
VIDEOS = [
    "FH102_02",
    "FH103_01",
    "FH303_01",
    "FH402_01",
    "FH509_01",
    "FH706_01",
    "FH803_01",
]
# ══════════════════════════════════════════════════════════════

In [ ]:
# ══════════════════════════════════════════════════════════════
class CameraTrapDataset(Dataset):
    def __init__(self, frames_folder, img_size=256, use_diff=False):
        self.use_diff    = use_diff
        self.frame_paths = sorted([
            os.path.join(frames_folder, f)
            for f in os.listdir(frames_folder)
            if f.endswith('.jpg')
        ])
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])
        mode = "diff" if use_diff else "raw"
        print(f"  Dataset: {len(self.frame_paths)} frames  [{mode} mode]")

    def __len__(self): return len(self.frame_paths)

    def __getitem__(self, idx):
        if self.use_diff and idx > 0:
            curr = cv2.cvtColor(cv2.imread(self.frame_paths[idx]),   cv2.COLOR_BGR2RGB)
            prev = cv2.cvtColor(cv2.imread(self.frame_paths[idx-1]), cv2.COLOR_BGR2RGB)
            img  = cv2.absdiff(curr, prev)
        else:
            img = cv2.cvtColor(cv2.imread(self.frame_paths[idx]), cv2.COLOR_BGR2RGB)
        return self.transform(img), self.frame_paths[idx]


class Autoencoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3,   64,  4, 2, 1), nn.ReLU(),
            nn.Conv2d(64,  128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Flatten(), nn.Linear(512*16*16, latent_dim)
        )
        self.decoder_fc = nn.Linear(latent_dim, 512*16*16)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64,  4, 2, 1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.ConvTranspose2d(64,  3,   4, 2, 1), nn.Tanh()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(self.decoder_fc(z).view(-1, 512, 16, 16))

    def anomaly_score(self, x):
        return torch.mean((x - self.forward(x))**2, dim=[1,2,3])


def train_model(dataset, device, name, output_folder):
    loader    = DataLoader(dataset, batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=4, pin_memory=True)
    model     = Autoencoder(LATENT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    losses    = []

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        for frames, _ in tqdm(loader, desc=f"    Epoch {epoch+1}/{EPOCHS}", leave=False):
            frames = frames.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(frames), frames)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        epoch_loss /= len(loader)
        losses.append(epoch_loss)
        print(f"    Epoch [{epoch+1}/{EPOCHS}]  Loss: {epoch_loss:.4f}")

        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(),
                       os.path.join(output_folder, f"{name}_epoch{epoch+1}.pth"))

    torch.save(model.state_dict(), os.path.join(output_folder, f"{name}_final.pth"))
    print(f"    Saved: {name}_final.pth")
    return model, losses


def score_frames(model, dataset, device):
    loader     = DataLoader(dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=4, pin_memory=True)
    all_scores = []
    all_paths  = []
    model.eval()
    with torch.no_grad():
        for frames, paths in tqdm(loader, desc="    Scoring", leave=False):
            frames = frames.to(device, non_blocking=True)
            all_scores.extend(model.anomaly_score(frames).cpu().numpy())
            all_paths.extend(paths)
    return np.array(all_scores), all_paths


def smooth_max(scores, window=2):
    smoothed = np.zeros_like(scores)
    for i in range(len(scores)):
        smoothed[i] = np.max(scores[max(0, i-window):min(len(scores), i+window+1)])
    return smoothed


def normalise(scores):
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)


def load_ground_truth(csv_path, video_name):
    df       = pd.read_csv(csv_path)
    df_video = df[df['Video'] == video_name]
    positive = set(f"frame_{(r['Frame']-1):05d}.jpg"
                   for _, r in df_video.iterrows() if r['Truth'] == 'Positive')
    negative = set(f"frame_{(r['Frame']-1):05d}.jpg"
                   for _, r in df_video.iterrows() if r['Truth'] == 'Negative')
    print(f"  Ground truth: {len(positive)} bird frames, {len(negative)} normal labelled")
    return positive, negative


def evaluate(scores, all_paths, positive_frames, negative_frames, video_name):
    threshold          = scores.mean() + THRESHOLD_STD * scores.std()
    labelled_indices   = []
    unlabelled_indices = []
    y_true, y_pred     = [], []

    for i, (score, path) in enumerate(zip(scores, all_paths)):
        fn         = os.path.basename(path)
        is_flagged = 1 if score > threshold else 0
        if fn in positive_frames:
            labelled_indices.append(i); y_true.append(1); y_pred.append(is_flagged)
        elif fn in negative_frames:
            labelled_indices.append(i); y_true.append(0); y_pred.append(is_flagged)
        else:
            unlabelled_indices.append(i)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if y_true.sum() == 0:
        print(f"  WARNING: no bird frames in labelled set for {video_name}")
        return None

    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0)
    auc = roc_auc_score(y_true, [scores[i] for i in labelled_indices])
    ap  = average_precision_score(y_true, [scores[i] for i in labelled_indices])

    total_flagged      = int((scores > threshold).sum())
    total_flag_rate    = total_flagged / len(all_paths)
    labelled_flag_rate = float(y_pred.sum()) / len(y_true) if len(y_true) > 0 else 0
    birds_found        = int(y_true[y_pred == 1].sum())
    workload_reduction = 1 - total_flag_rate
    bias_ratio         = labelled_flag_rate / total_flag_rate if total_flag_rate > 0 else 0

    return {
        'video': video_name, 'total_frames': len(all_paths),
        'bird_frames': int(y_true.sum()), 'labelled_frames': len(y_true),
        'unlabelled_frames': len(unlabelled_indices),
        'auc': auc, 'ap': ap, 'precision': p, 'recall': r, 'f1': f,
        'total_flagged': total_flagged, 'total_flag_rate': total_flag_rate,
        'labelled_flag_rate': labelled_flag_rate, 'bias_ratio': bias_ratio,
        'workload_reduction': workload_reduction, 'birds_found': birds_found,
        'threshold': threshold, 'y_true': y_true, 'y_pred': y_pred,
        'labelled_indices': labelled_indices,
        'unlabelled_indices': unlabelled_indices,
    }


def print_video_results(r):
    print(f"\n  {'='*60}")
    print(f"  RESULTS — {r['video']}")
    print(f"  {'='*60}")
    print(f"  Total frames        : {r['total_frames']:>7,}")
    print(f"  Bird frames         : {r['bird_frames']:>7,}")
    print(f"  Labelled frames     : {r['labelled_frames']:>7,}  ({r['labelled_frames']/r['total_frames']:.1%})")
    print(f"  Unlabelled frames   : {r['unlabelled_frames']:>7,}")
    print(f"")
    print(f"  ── ALL frames ──────────────────────────────────────")
    print(f"  Total flagged       : {r['total_flagged']:>7,}  ({r['total_flag_rate']:.1%} of all frames)")
    print(f"  Workload reduction  : {r['workload_reduction']:.1%}")
    print(f"  Birds found         : {r['birds_found']} / {r['bird_frames']}  (recall {r['recall']:.1%})")
    print(f"")
    print(f"  ── LABELLED frames (evaluation metrics) ────────────")
    print(f"  AUC-ROC             : {r['auc']:>7.3f}  ← main metric")
    print(f"  Avg Precision       : {r['ap']:>7.3f}")
    print(f"  Precision           : {r['precision']:>7.3f}")
    print(f"  Recall              : {r['recall']:>7.3f}")
    print(f"  F1 score            : {r['f1']:>7.3f}")
    print(f"  Threshold           : {r['threshold']:>7.4f}  (mean + {THRESHOLD_STD}×std)")
    print(f"  Bias ratio          : {r['bias_ratio']:>7.1f}x")
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    print(f"\n  Confusion matrix:")
    print(f"                    Predicted")
    print(f"                    Normal    Bird")
    print(f"  Actual Normal   {cm[0][0]:>8,}  {cm[0][1]:>6,}")
    print(f"  Actual Bird     {cm[1][0]:>8,}  {cm[1][1]:>6,}")
    print(f"  {'='*60}")


def save_anomaly_data(scores_combined, scores_raw_norm, scores_diff_norm,
                      all_paths, positive_frames, negative_frames,
                      results, output_folder):
    """
    Save everything needed for future second-stage classification (Step 9).

    Files saved:
      scores_combined.npy   — combined anomaly score per frame
      scores_raw_norm.npy   — normalised raw model score per frame
      scores_diff_norm.npy  — normalised diff model score per frame
      all_paths.npy         — frame filenames in time order
      anomalies.csv         — FLAGGED frames with scores + GT labels
      all_scores.csv        — ALL frames with scores

    anomalies.csv has placeholder columns for Step 9:
      classifier_label        ← fill in: BIRD / false_positive
      classifier_confidence   ← fill in: 0.0 to 1.0
      false_positive_reason   ← fill in: wind / sun / shadow / insect / camera_shake
    """
    threshold  = results['threshold']
    path_index = {p: i for i, p in enumerate(all_paths)}

    # numpy arrays
    np.save(os.path.join(output_folder, "scores_combined.npy"),  scores_combined)
    np.save(os.path.join(output_folder, "scores_raw_norm.npy"),  scores_raw_norm)
    np.save(os.path.join(output_folder, "scores_diff_norm.npy"), scores_diff_norm)
    np.save(os.path.join(output_folder, "all_paths.npy"), np.array(all_paths))

    # anomalies.csv — flagged frames only, ranked by score
    flagged_ranked = sorted(
        [(s, p) for s, p in zip(scores_combined, all_paths) if s > threshold],
        reverse=True)

    anomaly_rows = []
    for rank, (score, path) in enumerate(flagged_ranked, 1):
        fn        = os.path.basename(path)
        idx       = path_index[path]
        frame_num = int(fn.replace("frame_", "").replace(".jpg", ""))
        gt_label  = ("BIRD"    if fn in positive_frames else
                     "normal"  if fn in negative_frames else "unknown")
        anomaly_rows.append({
            'rank':                  rank,
            'filename':              fn,
            'path':                  path,
            'frame_num':             frame_num,
            'score_combined':        round(float(score), 6),
            'score_raw':             round(float(scores_raw_norm[idx]), 6),
            'score_diff':            round(float(scores_diff_norm[idx]), 6),
            'gt_label':              gt_label,
            'flagged':               1,
            # Step 9 columns — fill in during classification
            'classifier_label':      '',
            'classifier_confidence': '',
            'false_positive_reason': '',  # wind/sun/shadow/insect/camera_shake
        })

    pd.DataFrame(anomaly_rows).to_csv(
        os.path.join(output_folder, "anomalies.csv"), index=False)

    # all_scores.csv — every frame
    all_rows = []
    for score, path in zip(scores_combined, all_paths):
        fn        = os.path.basename(path)
        idx       = path_index[path]
        frame_num = int(fn.replace("frame_", "").replace(".jpg", ""))
        gt_label  = ("BIRD"    if fn in positive_frames else
                     "normal"  if fn in negative_frames else "unknown")
        all_rows.append({
            'filename':       fn,
            'frame_num':      frame_num,
            'score_combined': round(float(score), 6),
            'score_raw':      round(float(scores_raw_norm[idx]), 6),
            'score_diff':     round(float(scores_diff_norm[idx]), 6),
            'gt_label':       gt_label,
            'flagged':        1 if score > threshold else 0,
        })

    pd.DataFrame(all_rows).to_csv(
        os.path.join(output_folder, "all_scores.csv"), index=False)

    birds_in  = sum(1 for r in anomaly_rows if r['gt_label'] == 'BIRD')
    normal_in = sum(1 for r in anomaly_rows if r['gt_label'] == 'normal')
    unkn_in   = sum(1 for r in anomaly_rows if r['gt_label'] == 'unknown')
    print(f"  anomalies.csv  : {len(anomaly_rows):,} flagged  "
          f"(BIRD={birds_in}, normal={normal_in}, unknown={unkn_in})")
    print(f"  all_scores.csv : {len(all_rows):,} total frames")
    print(f"  numpy arrays   : scores_combined, raw, diff, all_paths saved")


def plot_video_results(scores, all_paths, positive_frames,
                       results, output_folder, video_name):
    bird_indices = [i for i, p in enumerate(all_paths)
                    if os.path.basename(p) in positive_frames]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8))
    fig.suptitle(
        f"Anomaly detection — {video_name}  |  "
        f"AUC={results['auc']:.3f}  Recall={results['recall']:.3f}  "
        f"Precision={results['precision']:.3f}", fontsize=12)

    ax1.plot(scores, linewidth=0.5, color='steelblue', alpha=0.8, label='score')
    ax1.axhline(y=results['threshold'], color='red', linestyle='--',
                linewidth=1, label=f"threshold ({results['threshold']:.4f})")
    for idx in bird_indices:
        ax1.axvline(x=idx, color='green', alpha=0.25, linewidth=0.5)
    ax1.plot([], [], color='green', alpha=0.5, linewidth=2,
             label=f"ground truth ({len(bird_indices)} frames)")
    ax1.set_ylabel("Anomaly score")
    ax1.set_xlabel("Frame number (time →)")
    ax1.set_title("Scores over time")
    ax1.legend(fontsize=8)

    y_true          = results['y_true']
    scores_labelled = scores[results['labelled_indices']]
    if (y_true==0).sum() > 0:
        ax2.hist(scores_labelled[y_true==0], bins=40, alpha=0.6,
                 color='steelblue', label='normal', density=True)
    if (y_true==1).sum() > 0:
        ax2.hist(scores_labelled[y_true==1], bins=20, alpha=0.6,
                 color='green', label='bird', density=True)
    ax2.axvline(x=results['threshold'], color='red', linestyle='--', linewidth=1.5)
    ax2.set_xlabel("Anomaly score")
    ax2.set_ylabel("Density")
    ax2.set_title("Score distribution — bird vs normal (labelled frames only)")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "results.png"), dpi=120)
    plt.close()


def print_final_comparison(all_results):
    print(f"\n{'='*95}")
    print(f"  FINAL COMPARISON — ALL VIDEOS  "
          f"(raw×{WEIGHT_RAW}+diff×{WEIGHT_DIFF}, latent={LATENT_DIM}, epochs={EPOCHS})")
    print(f"{'='*95}")
    print(f"  {'Video':<12} {'Frames':>8} {'Birds':>6} "
          f"{'AUC':>7} {'AP':>7} {'Prec':>7} {'Recall':>8} {'F1':>7} "
          f"{'Found':>8} {'Workload↓':>10}")
    print(f"  {'-'*88}")
    for r in all_results:
        print(f"  {r['video']:<12} {r['total_frames']:>8,} "
              f"{r['bird_frames']:>6} {r['auc']:>7.3f} "
              f"{r['ap']:>7.3f} {r['precision']:>7.3f} "
              f"{r['recall']:>8.3f} {r['f1']:>7.3f} "
              f"{r['birds_found']:>3}/{r['bird_frames']:<4} "
              f"{r['workload_reduction']:>9.1%}")
    print(f"  {'-'*88}")
    print(f"  {'AVERAGE':<12} {'':>8} {'':>6} "
          f"{np.mean([r['auc'] for r in all_results]):>7.3f} "
          f"{np.mean([r['ap']  for r in all_results]):>7.3f} "
          f"{np.mean([r['precision'] for r in all_results]):>7.3f} "
          f"{np.mean([r['recall'] for r in all_results]):>8.3f} "
          f"{np.mean([r['f1']  for r in all_results]):>7.3f} "
          f"{sum(r['birds_found'] for r in all_results):>3}/"
          f"{sum(r['bird_frames'] for r in all_results):<4} "
          f"{np.mean([r['workload_reduction'] for r in all_results]):>9.1%}")
    print(f"{'='*95}")


def plot_comparison(all_results, output_folder):
    videos  = [r['video']              for r in all_results]
    aucs    = [r['auc']                for r in all_results]
    recalls = [r['recall']             for r in all_results]
    precs   = [r['precision']          for r in all_results]
    f1s     = [r['f1']                 for r in all_results]
    birds   = [r['birds_found']        for r in all_results]
    total   = [r['bird_frames']        for r in all_results]
    wl      = [r['workload_reduction'] for r in all_results]
    flagged = [r['total_flagged']      for r in all_results]

    x = np.arange(len(videos))
    w = 0.2

    fig, axes = plt.subplots(4, 1, figsize=(14, 16))
    fig.suptitle(
        f"Multi-video comparison — v4 dual model  "
        f"(raw×{WEIGHT_RAW}+diff×{WEIGHT_DIFF}, latent={LATENT_DIM}, epochs={EPOCHS})",
        fontsize=12)

    for i, (vals, label, color) in enumerate([
        (aucs,    'AUC-ROC',   'steelblue'),
        (recalls, 'Recall',    'green'),
        (precs,   'Precision', 'orange'),
        (f1s,     'F1',        'purple'),
    ]):
        axes[0].bar(x+(i-1.5)*w, vals, w, label=label, color=color, alpha=0.8)
    for i, vals in enumerate([aucs, recalls, precs, f1s]):
        for j, v in enumerate(vals):
            axes[0].text(j+(i-1.5)*w, v+0.01, f"{v:.2f}",
                         ha='center', fontsize=7)
    axes[0].set_xticks(x); axes[0].set_xticklabels(videos)
    axes[0].set_ylabel("Score"); axes[0].set_ylim(0, 1.15)
    axes[0].set_title("AUC / Recall / Precision / F1 per video")
    axes[0].axhline(y=np.mean(aucs), color='steelblue', ls=':', alpha=0.5)
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3, axis='y')

    axes[1].bar(x, total, label='Total birds', color='lightcoral', alpha=0.8)
    axes[1].bar(x, birds, label='Found',       color='green',      alpha=0.8)
    for i, (b, t) in enumerate(zip(birds, total)):
        axes[1].text(i, t+0.1, f"{b}/{t}\n{b/t:.0%}",
                     ha='center', fontsize=9, fontweight='bold')
    axes[1].set_xticks(x); axes[1].set_xticklabels(videos)
    axes[1].set_ylabel("Frames")
    axes[1].set_title("Birds found vs total per video")
    axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

    axes[2].bar(x, flagged, color='steelblue', alpha=0.8)
    for i, (fl, fr) in enumerate(zip(flagged, [r['total_flag_rate'] for r in all_results])):
        axes[2].text(i, fl+1, f"{fl:,}\n({fr:.1%})", ha='center', fontsize=8)
    axes[2].set_xticks(x); axes[2].set_xticklabels(videos)
    axes[2].set_ylabel("Flagged frames")
    axes[2].set_title("Total flagged frames per video")
    axes[2].grid(True, alpha=0.3, axis='y')

    axes[3].bar(x, [w*100 for w in wl], color='teal', alpha=0.8)
    for i, w_val in enumerate(wl):
        axes[3].text(i, w_val*100+0.3, f"{w_val:.1%}", ha='center', fontsize=9)
    axes[3].set_xticks(x); axes[3].set_xticklabels(videos)
    axes[3].set_ylabel("Workload reduction (%)"); axes[3].set_xlabel("Video")
    axes[3].set_title("Manual review workload reduction per video")
    axes[3].set_ylim(0, 105); axes[3].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "comparison_plot.png"), dpi=150)
    plt.close()
    print(f"Comparison plot saved")


# ══════════════════════════════════════════════════════════════
# STEP 9 — PLACEHOLDER: Second-stage classifier
# Implement AFTER reviewing multi-video results.
# ══════════════════════════════════════════════════════════════
def run_classifier(output_folder, video_name):
    """
    PLACEHOLDER — implement after seeing multi-video results.

    Goal: reduce false positives by classifying each flagged frame as:
      BIRD / wind / sun_change / shadow / insect / camera_shake / other

    Input:  anomalies.csv (scores + frame paths + gt_label)
    Output: anomalies.csv with classifier columns filled in

    Methods to explore (after seeing results):
      1. Rule-based   — score_diff >> score_raw → likely wind/motion FP
      2. CNN          — train small ResNet on manually labelled subset
      3. CLIP         — zero-shot: "hummingbird" vs "wind" vs "shadow"
      4. Temporal     — consecutive flagged frames → real visit

    To activate: uncomment and implement the rule_based block below,
    or replace with your chosen method.
    """
    anomalies_path = os.path.join(output_folder, "anomalies.csv")
    if not os.path.exists(anomalies_path):
        return

    df = pd.read_csv(anomalies_path)
    print(f"\n  [Step 9 — classifier placeholder]")
    print(f"  {len(df)} flagged frames ready for classification")
    print(f"  Implement classifier here after reviewing multi-video results")
    print(f"  See run_classifier() docstring for suggested methods")

    # ── Rule-based starter (uncomment to try) ─────────────────
    # def rule_based(row):
    #     # high diff, low raw → temporal anomaly → likely wind
    #     if row['score_diff'] > 0.7 and row['score_raw'] < 0.3:
    #         return 'false_positive', 0.7, 'wind'
    #     # both high → genuine anomaly → likely bird
    #     if row['score_combined'] > 0.8:
    #         return 'BIRD', 0.8, ''
    #     return 'uncertain', 0.5, ''
    #
    # df[['classifier_label', 'classifier_confidence', 'false_positive_reason']] = \
    #     df.apply(lambda r: pd.Series(rule_based(r)), axis=1)
    # df.to_csv(anomalies_path, index=False)
    # print(f"  classifier_label counts:")
    # print(df['classifier_label'].value_counts().to_string())


# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════
def main():
    os.makedirs(OUTPUT_BASE, exist_ok=True)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else
        "mps"  if torch.backends.mps.is_available() else "cpu")
    print(f"\nDevice: {device}")
    if device.type == "cuda":
        print(f"GPU   : {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

    df_gt      = pd.read_csv(CSV_PATH)
    csv_videos = set(df_gt['Video'].unique())
    print(f"\nCSV: {len(df_gt)} labelled frames across {len(csv_videos)} videos")
    missing = [v for v in VIDEOS if v not in csv_videos]
    if missing:
        print(f"WARNING: not in CSV: {missing}")

    all_results = []

    for video_idx, video_name in enumerate(VIDEOS):
        print(f"\n{'#'*65}")
        print(f"  VIDEO {video_idx+1}/{len(VIDEOS)}: {video_name}")
        print(f"{'#'*65}")

        frames_folder = os.path.join(FRAMES_BASE, video_name)
        output_folder = os.path.join(OUTPUT_BASE, video_name)
        os.makedirs(output_folder, exist_ok=True)

        if not os.path.exists(frames_folder):
            print(f"  SKIP: frames folder not found: {frames_folder}")
            continue

        n_frames = len([f for f in os.listdir(frames_folder) if f.endswith('.jpg')])
        if n_frames == 0:
            print(f"  SKIP: no jpg frames in {frames_folder}")
            continue
        print(f"  Frames: {n_frames:,}")

        # ── RESUME CHECK ──────────────────────────────────────
        model_a_path = os.path.join(output_folder, "model_A_raw_final.pth")
        model_b_path = os.path.join(output_folder, "model_B_diff_final.pth")
        results_path = os.path.join(output_folder, "results.npy")
        already_done = (os.path.exists(model_a_path) and
                        os.path.exists(model_b_path) and
                        os.path.exists(results_path))

        if already_done:
            print(f"  ✓ Already done — loading saved results")
            results = np.load(results_path, allow_pickle=True).item()
            all_results.append(results)
            print_video_results(results)
            run_classifier(output_folder, video_name)
            continue

        positive_frames, negative_frames = load_ground_truth(CSV_PATH, video_name)
        if not positive_frames:
            print(f"  SKIP: no positive frames in CSV for {video_name}")
            continue

        with open(os.path.join(output_folder, "config.txt"), "w") as f:
            f.write(f"Video={video_name}\nTimestamp={TIMESTAMP}\n"
                    f"Frames={n_frames}\nLatent={LATENT_DIM}\n"
                    f"Epochs={EPOCHS}\nBatch={BATCH_SIZE}\n"
                    f"LR={LEARNING_RATE}\nThreshold={THRESHOLD_STD}std\n"
                    f"Weights=raw{WEIGHT_RAW}+diff{WEIGHT_DIFF}\n"
                    f"Device={device}\n")

        # Step 1
        print(f"\n  Step 1 — Loading datasets")
        dataset_raw  = CameraTrapDataset(frames_folder, IMG_SIZE, use_diff=False)
        dataset_diff = CameraTrapDataset(frames_folder, IMG_SIZE, use_diff=True)

        # Step 2
        print(f"\n  Step 2 — Training Model A (raw)")
        model_raw, losses_raw = train_model(
            dataset_raw, device, "model_A_raw", output_folder)

        # Step 3
        print(f"\n  Step 3 — Training Model B (diff)")
        model_diff, losses_diff = train_model(
            dataset_diff, device, "model_B_diff", output_folder)

        # loss curves
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(losses_raw,  lw=2, color='steelblue')
        ax1.set_title(f"Model A (raw) — {video_name}")
        ax2.plot(losses_diff, lw=2, color='purple')
        ax2.set_title(f"Model B (diff) — {video_name}")
        for ax in [ax1, ax2]:
            ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, "loss_curves.png"), dpi=120)
        plt.close()

        # Step 4
        print(f"\n  Step 4 — Scoring all frames")
        scores_raw,  all_paths = score_frames(model_raw,  dataset_raw,  device)
        scores_diff, _         = score_frames(model_diff, dataset_diff, device)

        # Step 5
        print(f"\n  Step 5 — Combining scores")
        scores_diff_smooth = smooth_max(scores_diff, SMOOTH_WINDOW)
        scores_raw_norm    = normalise(scores_raw)
        scores_diff_norm   = normalise(scores_diff_smooth)
        scores_combined    = WEIGHT_RAW * scores_raw_norm + WEIGHT_DIFF * scores_diff_norm
        print(f"  Raw   mean={scores_raw.mean():.4f}  std={scores_raw.std():.4f}")
        print(f"  Diff  mean={scores_diff.mean():.4f}  std={scores_diff.std():.4f}")
        print(f"  Combined mean={scores_combined.mean():.4f}  std={scores_combined.std():.4f}")

        # Step 6
        print(f"\n  Step 6 — Evaluating")
        results = evaluate(scores_combined, all_paths,
                           positive_frames, negative_frames, video_name)
        if results is None:
            continue

        print_video_results(results)
        all_results.append(results)

        # Step 7
        print(f"\n  Step 7 — Saving anomaly data")
        save_anomaly_data(
            scores_combined, scores_raw_norm, scores_diff_norm,
            all_paths, positive_frames, negative_frames,
            results, output_folder)

        # save results for resume
        np.save(os.path.join(output_folder, "results.npy"), results)
        print(f"  results.npy saved — resume will skip this video next run")

        # Step 8
        print(f"\n  Step 8 — Saving suspicious frame images")
        susp_folder = os.path.join(output_folder, "suspicious_frames")
        os.makedirs(susp_folder, exist_ok=True)
        flagged = sorted(
            [(s, p) for s, p in zip(scores_combined, all_paths)
             if s > results['threshold']], reverse=True)
        for i, (score, path) in enumerate(flagged):
            fn      = os.path.basename(path)
            is_bird = "BIRD" if fn in positive_frames else "normal"
            shutil.copy(path, os.path.join(
                susp_folder, f"rank{i+1:04d}_{is_bird}_{score:.4f}_{fn}"))
        print(f"  {len(flagged)} frames saved")

        # plot
        plot_video_results(scores_combined, all_paths, positive_frames,
                           results, output_folder, video_name)

        # Step 9 (placeholder)
        run_classifier(output_folder, video_name)

        # free GPU memory
        del model_raw, model_diff
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # Final comparison
    if not all_results:
        print("\nNo results — check FRAMES_BASE and CSV_PATH")
        return

    print_final_comparison(all_results)
    plot_comparison(all_results, OUTPUT_BASE)

    df_out = pd.DataFrame([{
        'video':               r['video'],
        'total_frames':        r['total_frames'],
        'bird_frames':         r['bird_frames'],
        'labelled_frames':     r['labelled_frames'],
        'auc':                 round(r['auc'],        3),
        'avg_precision':       round(r['ap'],         3),
        'precision':           round(r['precision'],  3),
        'recall':              round(r['recall'],     3),
        'f1':                  round(r['f1'],         3),
        'birds_found':         r['birds_found'],
        'total_flagged':       r['total_flagged'],
        'flag_rate_pct':       round(r['total_flag_rate']*100,    1),
        'workload_reduction_pct': round(r['workload_reduction']*100, 1),
        'bias_ratio':          round(r['bias_ratio'],  1),
        'threshold':           round(r['threshold'],   4),
        'epochs':              EPOCHS,
        'latent_dim':          LATENT_DIM,
        'weight_raw':          WEIGHT_RAW,
        'weight_diff':         WEIGHT_DIFF,
    } for r in all_results])

    csv_out = os.path.join(OUTPUT_BASE, f"comparison_{TIMESTAMP}.csv")
    df_out.to_csv(csv_out, index=False)

    print(f"\nComparison CSV : {csv_out}")
    print(f"All outputs in : {OUTPUT_BASE}")
    print(f"\n── Next steps ─────────────────────────────────────")
    print(f"  1. Review comparison_plot.png")
    print(f"  2. Check anomalies.csv per video — look for FP patterns")
    print(f"  3. If AUC varies across videos → investigate why")
    print(f"  4. Then implement Step 9 classifier in run_classifier()")


if __name__ == "__main__":
    main()


Device: cuda
GPU   : NVIDIA A100 80GB PCIe MIG 2g.20gb
Memory: 20.9 GB

CSV: 39621 labelled frames across 70 videos

#################################################################
  VIDEO 1/7: FH102_02
#################################################################
  Frames: 21,593
  Ground truth: 33 bird frames, 925 normal labelled

  Step 1 — Loading datasets
  Dataset: 21593 frames  [raw mode]
  Dataset: 21593 frames  [diff mode]

  Step 2 — Training Model A (raw)


    Epoch 1/30:   0%|          | 0/338 [00:00<?, ?it/s]

### Chat history
https://claude.ai/share/48998536-f139-45da-8afb-7859e67ae9e9